In [0]:
-----------1. Extracting and  sorting the userprofile table-----

SELECT *
FROM workspace.default.userprofiles
LIMIT 1000;
----------1.1. COUNTING THE NUMBER OF USERS
SELECT COUNT(DISTINCT UserID ) AS NO_OF_VIEWERS
FROM workspace.default.userprofiles; 

----------1.2. FILTERING AND SORTING USERPROFILE TABLE-----------

SELECT IFNULL(UserID,'0') AS USERID,
       IFNULL(Gender,'NO GENDER') AS GENDER,
       IFNULL(Age,'0') AS AGE,
       IFNULL(Race,'NO RACE') AS RACE,
       IFNULL(Province,'NO PROVINCE') AS PROVINCE
FROM workspace.default.userprofiles;     

----------1.3. VIEWERSHIP COUNT BY GENDER AND AGE------------------------

SELECT IFNULL(Gender, 'NO GENDER')  AS GENDER,
       IFNULL(Age, 0)   AS AGE,
       COUNT(*)     AS GENDER_AGE_COUNT
FROM workspace.default.userprofiles
GROUP BY Gender, Age
ORDER BY GENDER_AGE_COUNT ASC;

---------1.4. Most watched channels-----------

-- MOST WATCHED CHANNELS BY NUMBER OF SESSIONS
SELECT IFNULL(Channel2, 'NO CHANNEL')   AS CHANNEL,
       COUNT(*)   AS TOTAL_SESSIONS,
       COUNT(DISTINCT UserID0) AS UNIQUE_VIEWERS,
       ROUND(SUM((HOUR(Duration2) * 60) + MINUTE(Duration2) + (SECOND(Duration2) / 60.0)), 2)  AS TOTAL_MINUTES_WATCHED,
       ROUND(AVG((HOUR(Duration2) * 60) + MINUTE(Duration2) + (SECOND(Duration2) / 60.0)), 2)  AS AVG_SESSION_MINUTES
FROM workspace.default.viewerships
GROUP BY Channel2
ORDER BY TOTAL_SESSIONS DESC;
  
-------- 2. Extracting and  sorting the viewership table......

SELECT *
FROM workspace.default.viewerships
LIMIT 1000;
--------------2.1 FILTERING AND SORTING VIEWWERSHIP TABLE-----------
SELECT DISTINCT
       IFNULL(UserID0, '0')  AS USERID,
       IFNULL(Channel2, 'NO CHANNEL') AS CHANNELS,
       IFNULL(Duration2, 'NO DURATION') AS DURATION,
       IFNULL(RecordDate2, 'NO RECORD DATE') AS RECORD_DATE,
       YEAR(RecordDate2) AS YEAR_RECORDED,
       MONTH(RecordDate2) AS MONTH_RECORDED,
       DAYNAME(RecordDate2) AS DAY_RECORDED
FROM workspace.default.viewerships;
---------------------------------------

------------2.2. NUMBER OF CHANNELS-----
SELECT COUNT( DISTINCT Channel2 ) AS NO_OF_CHANNELS
FROM workspace.default.viewerships;

--------------------3. COMBINING BOTH TABLEs into 1--------
SELECT COALESCE(U.UserID, V.UserID0, '0')  AS USER_ID,
           IFNULL(U.Gender, 'NO GENDER')  AS GENDER,
           IFNULL(U.Age, 'NO AGE ') AS AGE,
           IFNULL(U.Race, 'NO RACE')AS RACE,
           IFNULL(U.Province, 'NO PROVINCE') AS PROVINCE,
           IFNULL(V.Channel2, 'NO CHANNEL') AS CHANNEL,

--------- CONVERTING DATE FORMAT --------------------------------
           DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2), 'yyyy-MM-dd') AS RECORD_DATE,
           DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2), 'HH:mm:ss') AS RECORD_TIME_SAST,
           DAYNAME(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2))  AS DAY_NAME,
---------- DURATION BREAKDOWN-------------------------------------
           DATE_FORMAT(V.Duration2, 'HH:mm:ss') AS DURATION,
           ROUND(((HOUR(V.Duration2) * 60) + MINUTE(V.Duration2) + (SECOND(V.Duration2) / 60.0)), 2) AS DURATION_TOTAL_MINUTES
      
    FROM workspace.default.userprofiles AS U
    LEFT JOIN workspace.default.viewerships AS V
    ON U.UserID = V.UserID0;

--------4. USING BOTH TABLES AS BASE TO ANALYSE FURTHER---------
WITH BASE AS (

    SELECT COALESCE(CAST(U.UserID AS STRING), CAST(V.UserID0 AS STRING), '0')                       AS USER_ID,
           IFNULL(U.Gender, 'NO GENDER')                                                            AS GENDER,
           IFNULL(U.Age, 0)                                                                         AS AGE,
           IFNULL(U.Race, 'NO RACE')                                                                AS RACE,
           IFNULL(U.Province, 'NO PROVINCE')                                                        AS PROVINCE,
           IFNULL(V.Channel2, 'NO CHANNEL')                                                         AS CHANNEL,

           -- CONVERTING DATE FORMAT
           DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2), 'yyyy-MM-dd') AS RECORD_DATE,
           DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2), 'HH:mm:ss')   AS RECORD_TIME_SAST,
           MONTHNAME(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2))                 AS MONTH_NAME,
           DAYNAME(CONVERT_TIMEZONE('UTC', 'Africa/Johannesburg', V.RecordDate2))                   AS DAY_NAME,

           -- DURATION BREAKDOWN
           DATE_FORMAT(V.Duration2, 'HH:mm:ss')                                                     AS DURATION,
           ROUND(((HOUR(V.Duration2) * 60) + MINUTE(V.Duration2) + (SECOND(V.Duration2) / 60.0)), 2) AS DURATION_TOTAL_MINUTES

    FROM workspace.default.userprofiles AS U
    LEFT JOIN workspace.default.viewerships AS V
    ON CAST(U.UserID AS STRING) = CAST(V.UserID0 AS STRING)

)

SELECT USER_ID,
       GENDER,
       AGE,
       CASE
           WHEN AGE = 0            THEN 'UNKNOWN'
           WHEN AGE < 13           THEN 'CHILD'
           WHEN AGE >= 13 AND AGE < 18 THEN 'ADOLESCENT TEEN'
           WHEN AGE >= 18 AND AGE < 25 THEN 'YOUNG ADULT'
           WHEN AGE >= 25 AND AGE < 45 THEN 'ADULT'
           WHEN AGE >= 45 AND AGE < 65 THEN 'MIDDLE-AGED ADULT'
           WHEN AGE >= 65          THEN 'SENIOR CITIZEN'
           ELSE 'OTHER'
       END                                                                                           AS AGE_CLASSIFICATION,
       RACE,
       PROVINCE,
       CHANNEL,
       RECORD_DATE,
       MONTH_NAME,
       RECORD_TIME_SAST,
       CASE
           WHEN RECORD_TIME_SAST BETWEEN '05:00:00' AND '11:59:59' THEN 'MORNING SLOT'
           WHEN RECORD_TIME_SAST BETWEEN '12:00:00' AND '17:59:59' THEN 'AFTERNOON SLOT'
           WHEN RECORD_TIME_SAST >= '18:00:00'                     THEN 'EVENING SLOT'
           WHEN RECORD_TIME_SAST < '05:00:00'                      THEN 'MIDNIGHT OWL'
           ELSE 'MIDNIGHT OWL'
       END                                                                                           AS TIME_SLOT,
       DAY_NAME,
       DURATION,
       DURATION_TOTAL_MINUTES

FROM BASE
ORDER BY RECORD_DATE DESC;
